# Dự đoán khả năng vỡ nợ của khách hàng sử dụng thẻ tín dụng
**Bài tập nhóm cuối kỳ – Học phần Nhận dạng mẫu**

## Phần 1. Bài toán, dữ liệu và phân tích sơ bộ

**Bài toán.** Mỗi khách hàng được biểu diễn bằng vector đặc trưng $x \in \mathbb{R}^{23}$ (hạn mức, nhân khẩu học, lịch sử trả nợ, dư nợ và số tiền thanh toán từ tháng 4 đến tháng 9/2005). Cần phân loại khách hàng vào một trong hai lớp: $\omega_1$ – vỡ nợ trong tháng 10/2005, hoặc $\omega_2$ – không vỡ nợ. Đây là bài toán phân loại nhị phân có mất cân bằng lớp.

**Nguồn dữ liệu.**
- Default of Credit Card Clients, UCI Machine Learning Repository, mã 350: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
- Tác giả: I-Cheng Yeh, đóng góp ngày 25/01/2016. DOI: 10.24432/C55S3H. Giấy phép CC BY 4.0.
- Bài báo giới thiệu: I.-C. Yeh and C.-H. Lien, "The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients," *Expert Systems with Applications*, vol. 36, no. 2, pp. 2473–2480, 2009.

**Dữ liệu đầu vào trên Kaggle.** Thêm một trong hai nguồn qua **Add Input**:
- Dataset *Default of Credit Card Clients Dataset* của UCI Machine Learning (`uciml/default-of-credit-card-clients-dataset`, tệp `UCI_Credit_Card.csv`), hoặc
- Tệp gốc `default of credit card clients.xls` tải từ trang UCI.

Notebook tự tìm tệp dữ liệu trong `/kaggle/input`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 30)

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

TARGET = "default"
PAY = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL = [f"BILL_AMT{i}" for i in range(1, 7)]
PAYAMT = [f"PAY_AMT{i}" for i in range(1, 7)]
MONTHS = ["T9", "T8", "T7", "T6", "T5", "T4"]  # cột 1 = tháng 9/2005 ... cột 6 = tháng 4/2005
CLASS_NAMES = {0: "Không vỡ nợ", 1: "Vỡ nợ"}
PALETTE = {"Không vỡ nợ": "#4C72B0", "Vỡ nợ": "#DD5555"}

## 1. Đọc dữ liệu

In [ ]:
def find_data():
    roots = [Path("/kaggle/input"), Path("../data/raw"), Path("data/raw")]
    for root in roots:
        if root.exists():
            for pattern in ("UCI_Credit_Card.csv", "default of credit card clients.xls"):
                hits = list(root.rglob(pattern))
                if hits:
                    return hits[0]
    raise FileNotFoundError("Chưa thêm dữ liệu: vào Add Input và chọn bộ Default of Credit Card Clients.")


path = find_data()
if path.suffix == ".csv":
    df = pd.read_csv(path).rename(columns={"default.payment.next.month": TARGET})
else:
    df = pd.read_excel(path, header=1).rename(columns={"default payment next month": TARGET})
df = df.astype({c: "int64" for c in df.columns})
df["Lớp"] = df[TARGET].map(CLASS_NAMES)

print("Nguồn:", path)
print("Kích thước:", df.shape)
df.head()

## 2. Tổng quan và chất lượng dữ liệu
Bộ dữ liệu có 30.000 khách hàng, 23 đặc trưng đầu vào, không có giá trị thiếu. Ngoài ra cần kiểm tra các mã không được mô tả (`EDUCATION` = 0, 5, 6; `MARRIAGE` = 0), dư nợ âm và các dòng trùng lặp.

In [ ]:
feats = df.drop(columns=["ID", TARGET, "Lớp"])
overview = pd.Series({
    "Số mẫu": len(df),
    "Số đặc trưng": feats.shape[1],
    "Giá trị thiếu": int(df.isna().sum().sum()),
    "Dòng trùng lặp (bỏ ID)": int(df.drop(columns=["ID", "Lớp"]).duplicated().sum()),
    "Số khách vỡ nợ": int(df[TARGET].sum()),
    "Tỷ lệ vỡ nợ": round(df[TARGET].mean(), 4),
    "EDUCATION mã 0/5/6": int(df["EDUCATION"].isin([0, 5, 6]).sum()),
    "MARRIAGE mã 0": int((df["MARRIAGE"] == 0).sum()),
    "BILL_AMT âm ít nhất 1 tháng": int((df[BILL] < 0).any(axis=1).sum()),
    "BILL_AMT = 0 cả 6 tháng": int((df[BILL] == 0).all(axis=1).sum()),
})
overview.to_frame("Giá trị")

In [ ]:
for c in ["SEX", "EDUCATION", "MARRIAGE", "PAY_0", "PAY_2", "PAY_6"]:
    print(c, df[c].value_counts().sort_index().to_dict())

### 2.1. Phân bố nhãn

In [ ]:
counts = df["Lớp"].value_counts().reindex(CLASS_NAMES.values())
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values, color=[PALETTE[c] for c in counts.index])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / len(df):.1%})", ha="center", va="bottom")
ax.set_ylim(0, counts.max() * 1.18)
ax.set_ylabel("Số khách hàng")
ax.set_title("Phân bố nhãn (tháng 10/2005)")
plt.show()

Tỷ lệ vỡ nợ là 22,12%. Một bộ phân loại luôn dự đoán "không vỡ nợ" vẫn đạt độ chính xác 77,88%, vì vậy accuracy không được dùng làm độ đo chính.

### 2.2. Thống kê mô tả

In [ ]:
desc = feats.describe().T
desc["skew"] = feats.skew()
desc.round(2)

Các đặc trưng tiền tệ (`BILL_AMT`, `PAY_AMT`) lệch phải rất mạnh, cần chuẩn hoá và biến đổi log trước khi dùng các mô hình nhạy với thang đo.

## 3. Lịch sử trả nợ

In [ ]:
rate = df.groupby("PAY_0")[TARGET].agg(["count", "mean"])
late = (df[PAY] > 0).sum(axis=1)
g = df.groupby(late)[TARGET].agg(["count", "mean"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, tab, xlabel, title in [
    (axes[0], rate, "PAY_0 (-2, -1, 0 = không trễ; k = trễ k tháng)", "Tỷ lệ vỡ nợ theo trạng thái trả nợ tháng 9"),
    (axes[1], g, "Số tháng trễ hạn trong 6 tháng", "Tỷ lệ vỡ nợ theo số tháng trễ hạn"),
]:
    ax.bar(tab.index.astype(str), tab["mean"], color="#DD5555")
    for i, (n, r) in enumerate(zip(tab["count"], tab["mean"])):
        ax.text(i, r, f"{r:.0%}\nn={n:,}", ha="center", va="bottom", fontsize=8)
    ax.axhline(df[TARGET].mean(), ls="--", c="gray", lw=1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Tỷ lệ vỡ nợ")
    ax.set_title(title)
plt.tight_layout()
plt.show()
g.rename(columns={"count": "Số khách", "mean": "Tỷ lệ vỡ nợ"}).T.round(3)

Khách hàng không trễ hạn ở tháng 9 có tỷ lệ vỡ nợ 13–17%, trong khi khách trễ 2 tháng có tỷ lệ 69%. Tỷ lệ vỡ nợ tăng gần như đơn điệu theo số tháng trễ hạn, từ 11,7% lên 70,3%.

## 4. Diễn biến hành vi qua sáu tháng

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, cols, title in [(axes[0], PAY, "Trạng thái trả nợ TB"),
                        (axes[1], BILL, "Dư nợ sao kê TB (NT$)"),
                        (axes[2], PAYAMT, "Số tiền đã trả TB (NT$)")]:
    m = df.groupby("Lớp")[cols].mean()
    for cls, row in m.iterrows():
        ax.plot(MONTHS[::-1], row.values[::-1], marker="o", label=cls, color=PALETTE[cls])
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

Trạng thái trả nợ của nhóm vỡ nợ xấu đi liên tục từ tháng 4 đến tháng 9, còn nhóm không vỡ nợ gần như ổn định. Dư nợ của hai nhóm gần như song song; khác biệt lớn nằm ở số tiền thanh toán (khoảng 3.200–3.400 NT$ so với 5.200–6.600 NT$ mỗi tháng).

## 5. Hạn mức tín dụng và tuổi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sns.histplot(data=df, x="LIMIT_BAL", hue="Lớp", stat="density", common_norm=False,
             bins=50, element="step", palette=PALETTE, ax=axes[0])
axes[0].set_title("Phân bố hạn mức tín dụng (NT$)")
sns.histplot(data=df, x="AGE", hue="Lớp", stat="density", common_norm=False,
             bins=range(20, 80, 2), element="step", palette=PALETTE, ax=axes[1])
axes[1].set_title("Phân bố tuổi")
plt.tight_layout()
plt.show()

age_bin = pd.cut(df["AGE"], [20, 25, 30, 35, 40, 45, 50, 60, 80], right=False)
lim_bin = pd.qcut(df["LIMIT_BAL"], 5)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, b, title in [(axes[0], age_bin, "nhóm tuổi"), (axes[1], lim_bin, "ngũ phân vị hạn mức")]:
    r = df.groupby(b, observed=True)[TARGET].mean()
    ax.bar(r.index.astype(str), r.values, color="#DD5555")
    ax.axhline(df[TARGET].mean(), ls="--", c="gray", lw=1)
    ax.set_title(f"Tỷ lệ vỡ nợ theo {title}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for col in ["LIMIT_BAL", "AGE"] + BILL + PAYAMT:
    a, b = df.loc[df[TARGET] == 0, col], df.loc[df[TARGET] == 1, col]
    rows.append({"Đặc trưng": col, "Trung vị (không vỡ nợ)": a.median(),
                 "Trung vị (vỡ nợ)": b.median(), "p-value Mann–Whitney": stats.mannwhitneyu(a, b).pvalue})
pd.DataFrame(rows)

Hạn mức càng thấp thì rủi ro càng cao (31,8% ở nhóm thấp nhất so với 13,8% ở nhóm cao nhất). Tuổi có quan hệ hình chữ U: rủi ro cao ở nhóm 20–24 và từ 60 tuổi, thấp nhất ở 30–34, trong khi trung vị tuổi hai lớp bằng nhau (p = 0,37). Đây là quan hệ phi tuyến mà mô hình tuyến tính khó nắm bắt.

## 6. Các đặc trưng nhân khẩu học

In [ ]:
labels = {
    "SEX": {1: "Nam", 2: "Nữ"},
    "EDUCATION": {0: "0 (lạ)", 1: "Sau ĐH", 2: "Đại học", 3: "THPT", 4: "Khác", 5: "5 (lạ)", 6: "6 (lạ)"},
    "MARRIAGE": {0: "0 (lạ)", 1: "Đã kết hôn", 2: "Độc thân", 3: "Khác"},
}
chi_rows = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (col, mp) in zip(axes, labels.items()):
    r = df.groupby(col)[TARGET].agg(["count", "mean"]).rename(index=mp)
    ax.bar(r.index.astype(str), r["mean"], color="#DD5555")
    for i, (n, v) in enumerate(zip(r["count"], r["mean"])):
        ax.text(i, v, f"{v:.1%}\nn={n:,}", ha="center", va="bottom", fontsize=8)
    ax.axhline(df[TARGET].mean(), ls="--", c="gray", lw=1)
    ax.set_ylim(0, r["mean"].max() * 1.3)
    ax.set_title(f"Tỷ lệ vỡ nợ theo {col}")
    ax.tick_params(axis="x", rotation=30)
    chi2, p, *_ = stats.chi2_contingency(pd.crosstab(df[col], df[TARGET]))
    chi_rows.append({"Đặc trưng": col, "chi2": round(chi2, 2), "p-value": p})
plt.tight_layout()
plt.show()
pd.DataFrame(chi_rows)

Cả ba đặc trưng đều có liên hệ có ý nghĩa thống kê với nhãn, nhưng mức chênh lệch chỉ vài điểm phần trăm. Với cỡ mẫu lớn, p-value rất nhỏ ngay cả khi hiệu ứng yếu. Việc dùng các đặc trưng này cho quyết định tín dụng cũng đặt ra vấn đề công bằng.

## 7. Phân tích tương quan (Spearman)

In [ ]:
corr = df.drop(columns=["ID", "Lớp"]).corr(method="spearman")
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Ma trận tương quan Spearman")
plt.show()

with_target = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(with_target.index[::-1], with_target.values[::-1],
        color=["#DD5555" if v > 0 else "#4C72B0" for v in with_target.values[::-1]])
ax.set_title("Tương quan Spearman của từng đặc trưng với nhãn")
plt.show()
corr.loc[BILL, BILL].round(3)

Không đặc trưng đơn lẻ nào tương quan mạnh với nhãn (cao nhất là `PAY_0` với ρ = 0,29). Nhóm `BILL_AMT` có đa cộng tuyến mạnh (0,73–0,91) nhưng gần như không tương quan với nhãn.

## 8. Mức độ chồng lấn giữa hai lớp (PCA và Fisher LDA)

In [ ]:
X = StandardScaler().fit_transform(feats)
y = df[TARGET].values
pca = PCA().fit(X)
Z = pca.transform(X)[:, :2]
z_lda = LinearDiscriminantAnalysis(n_components=1).fit(X, y).transform(X).ravel()

idx = np.random.default_rng(42).choice(len(df), 6000, replace=False)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for cls in (0, 1):
    m = y[idx] == cls
    axes[0].scatter(Z[idx][m, 0], Z[idx][m, 1], s=4, alpha=0.4, color=PALETTE[CLASS_NAMES[cls]], label=CLASS_NAMES[cls])
axes[0].set_xlim(np.percentile(Z[:, 0], [0.5, 99.5]))
axes[0].set_ylim(np.percentile(Z[:, 1], [0.5, 99.5]))
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[0].set_title("PCA 2 chiều (mẫu 6.000 điểm)")
axes[0].legend(markerscale=4)

cum = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, len(cum) + 1), cum, marker="o")
axes[1].axhline(0.9, ls="--", c="gray")
axes[1].set_xlabel("Số thành phần chính")
axes[1].set_title("Phương sai tích luỹ của PCA")

for cls in (0, 1):
    sns.kdeplot(z_lda[y == cls], ax=axes[2], fill=True, color=PALETTE[CLASS_NAMES[cls]],
                label=CLASS_NAMES[cls], clip=np.percentile(z_lda, [0.5, 99.5]))
axes[2].set_title("Chiếu Fisher LDA 1 chiều")
axes[2].legend()
plt.tight_layout()
plt.show()
print("Số thành phần chính đạt 90% phương sai:", int(np.argmax(cum >= 0.9) + 1))

Hai lớp chồng lấn nhiều trong không gian đặc trưng, nên sai số Bayes của bài toán không nhỏ. Cải tiến cần đến từ đặc trưng tốt hơn, mô hình phi tuyến và chọn ngưỡng quyết định theo chi phí.

## 9. Đặc trưng phái sinh: tỷ lệ sử dụng hạn mức

In [ ]:
util = (df["BILL_AMT1"] / df["LIMIT_BAL"]).clip(-0.5, 1.5)
fig, ax = plt.subplots(figsize=(6.5, 4))
sns.histplot(x=util, hue=df["Lớp"], stat="density", common_norm=False, bins=60,
             element="step", palette=PALETTE, ax=ax)
ax.set_xlabel("BILL_AMT1 / LIMIT_BAL")
ax.set_title("Tỷ lệ sử dụng hạn mức tháng 9")
plt.show()
df.groupby(pd.cut(util, [-0.5, 0, 0.25, 0.5, 0.75, 1.0, 1.5]), observed=True)[TARGET].agg(["count", "mean"]).T.round(3)

Khác với dư nợ gốc, tỷ lệ sử dụng hạn mức cho thấy xu hướng rõ: dùng càng gần hết hạn mức thì rủi ro càng cao (16,8% ở mức dưới 25% so với 30,1% khi vượt hạn mức).

## 10. Tổng kết và định hướng

| Phát hiện | Định hướng |
|---|---|
| Mất cân bằng lớp (22% vỡ nợ) | ROC-AUC, PR-AUC, Recall, F1; chia có phân tầng; trọng số lớp hoặc SMOTE |
| Lịch sử trả nợ là nhóm mạnh nhất | Thêm số tháng trễ, mức trễ lớn nhất, xu hướng trễ |
| `BILL_AMT` đa cộng tuyến, ít thông tin | Tỷ lệ sử dụng hạn mức, tỷ lệ thanh toán, hoặc PCA |
| `PAY_AMT` lệch mạnh | Biến đổi log(1 + x) |
| Tuổi có quan hệ hình chữ U | Mô hình dạng cây/boosting hoặc chia nhóm tuổi |
| Mã lạ ở `EDUCATION`, `MARRIAGE` | Gộp vào nhóm "Khác" |
| Hai lớp chồng lấn nhiều | Chọn ngưỡng theo quy tắc rủi ro tối thiểu Bayes |